# Project Mu — Colab dossier pilot
Run cells in order on your A100 40 GB. This loads an open model into Colab; no Gemini API key, database access or Google Drive mounting is required. Upload only the supplied `colab_matchup_input_v2.json`. This is dossier v2; the old input is rejected before model loading.

Candidate: [Qwen3-32B-AWQ](https://huggingface.co/Qwen/Qwen3-32B-AWQ), 4-bit weights, non-thinking mode. GPU compatibility and throughput still require this actual run. This is one article, not a production batch.

Record your starting unit balance before connecting. Previously observed: **200 units; 6.73 units/hour**. The runtime panel is authoritative. Model downloads and loading consume session time too. Finish by downloading the result bundle, then Runtime → Disconnect and delete runtime.


In [ ]:
import time, subprocess, json, hashlib, pathlib, sys, importlib.metadata
SESSION_START = time.time()
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],text=True))
START_UNITS = 200.0  # edit to the balance shown now
UNITS_PER_HOUR = 6.73  # edit if the runtime panel shows a different rate


## 1. Install the inference runtime
This may take several minutes. After installation, use Runtime → Restart session once (not Disconnect and delete). Rerun the first timing cell, skip this installation cell, then continue below. This clears any previously imported library versions.

In [ ]:
%pip install -q 'vllm==0.10.2' 'transformers==4.55.2' 'tokenizers==0.21.4' 'huggingface-hub==0.34.4'
print('Installation complete. Runtime → Restart session; rerun the first timing cell, then skip this cell.')


## 2. Upload input and check context before downloading weights

In [ ]:
from google.colab import files
expected = {'vllm':'0.10.2','transformers':'4.55.2','tokenizers':'0.21.4','huggingface-hub':'0.34.4'}
for package, version in expected.items():
    try:
        actual = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(f'{package} is not installed in this runtime. Run the installation cell above, then Runtime → Restart session (not Disconnect and delete runtime), rerun the first timing cell and continue here.') from None
    if actual != version:
        raise RuntimeError(f'{package} {actual}, expected {version}. Run the installation cell and restart session.')
import transformers, tokenizers, huggingface_hub
if transformers.__version__ != expected['transformers'] or tokenizers.__version__ != expected['tokenizers']:
    raise RuntimeError('Old libraries remain imported: Runtime → Restart session, rerun the first timing cell, then continue here.')
print('Dependency versions verified:',expected)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only colab_matchup_input_v2.json')
input_bytes = next(iter(uploaded.values()))
job = json.loads(input_bytes)
assert job['schema_version'] == 1
if job.get('dossier_version') != 2:
    raise RuntimeError('Wrong input: this is the original dossier. Upload colab_matchup_input_v2.json; do not load the model yet.')
print('DOSSIER VERSION:',job['dossier_version'])
print(job['assignment_header'])
assert [m['role'] for m in job['messages']] == ['system','user']
MODEL_ID = 'Qwen/Qwen3-32B-AWQ'
MAX_MODEL_LEN = 40960
MAX_NEW_TOKENS = 4096
from huggingface_hub import model_info
from transformers import AutoTokenizer
# Resolve one immutable model commit for tokenizer and weights.
MODEL_REVISION = job.get('model_revision') or model_info(MODEL_ID).sha
TOKENIZER_START = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
if not hasattr(tokenizer, 'all_special_tokens_extended'):
    raise RuntimeError('Tokenizer API incompatible with pinned vLLM; stop before loading weights.')
prompt_ids = tokenizer.apply_chat_template(job['messages'], tokenize=True, add_generation_prompt=True, enable_thinking=False, return_dict=False)
print('Prompt tokens:',len(prompt_ids),'Output allowance:',MAX_NEW_TOKENS)
if len(prompt_ids) + MAX_NEW_TOKENS > MAX_MODEL_LEN:
    raise RuntimeError('Input exceeds the configured context. Stop and send the token count; nothing will be truncated.')
print('Input fits configured context. Proceed to load weights.')


## 3. Load the model
The initial download is large. Keep the notebook open. If loading fails, send the error rather than repeatedly retrying. No paid API fallback is configured.

In [ ]:
import torch
from vllm import LLM, SamplingParams
if not torch.cuda.is_available():
    raise RuntimeError('No GPU connected')
if torch.cuda.get_device_properties(0).total_memory < 39 * 1024**3:
    raise RuntimeError('This configuration expects the observed A100 40 GB; stop if hardware changed')
LOAD_START = time.time()
llm = LLM(model=MODEL_ID, revision=MODEL_REVISION, tokenizer_revision=MODEL_REVISION,
          quantization='awq', dtype='half', tensor_parallel_size=1,
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          max_num_seqs=1, max_num_batched_tokens=2048,
          enable_chunked_prefill=True, enforce_eager=True,
          trust_remote_code=False, seed=42)
LOAD_SECONDS = time.time() - LOAD_START
print('Model load seconds:',round(LOAD_SECONDS,1))
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.used,memory.total','--format=csv'],text=True))


## 4. Generate and save one article
Creative scenes, quotes, access and unrecorded plays are allowed when they do not contradict canonical facts. A completed generation is not a factual approval.

In [ ]:
params = SamplingParams(temperature=0.7, top_p=0.8, top_k=20, min_p=0,
                        repetition_penalty=1.0, max_tokens=MAX_NEW_TOKENS, seed=42)
GEN_START = time.time()
outputs = llm.generate([{'prompt_token_ids':prompt_ids}], params)
GEN_SECONDS = time.time() - GEN_START
answer = outputs[0].outputs[0]
article = answer.text
if not article.strip():
    raise RuntimeError('Empty article; inspect output rather than publishing')
output_dir = pathlib.Path('/content/project_mu_result')
output_dir.mkdir(exist_ok=True)
(output_dir/'article.md').write_text(article)
report = dict(model=MODEL_ID,revision=MODEL_REVISION,input_sha256=hashlib.sha256(input_bytes).hexdigest(),
              dossier_version=job.get('dossier_version',1),assignment=job['assignment'],source_packet_sha256=job['source_packet_sha256'],
              prompt_tokens=len(prompt_ids),output_tokens=len(answer.token_ids),finish_reason=answer.finish_reason,
              output_capped=answer.finish_reason=='length',load_seconds=LOAD_SECONDS,generation_seconds=GEN_SECONDS,
              session_elapsed_seconds=time.time()-SESSION_START,
              estimated_units_since_first_cell=(time.time()-SESSION_START)/3600*UNITS_PER_HOUR,
              starting_units=START_UNITS,displayed_units_per_hour=UNITS_PER_HOUR,
              max_model_len=MAX_MODEL_LEN,sampling=params.__repr__(),
              packages={p:importlib.metadata.version(p) for p in ['vllm','torch','transformers']},
              gpu_snapshot=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.used,memory.total','--format=csv'],text=True),
              factual_status='unreviewed',limitations=['Unit estimate excludes time before first cell and is not account balance reconciliation',
              'nvidia-smi snapshot is not measured session peak VRAM','First model generation; no warm-throughput claim'])
(output_dir/'report.json').write_text(json.dumps(report,indent=2))
(output_dir/'input.json').write_bytes(input_bytes)
print(article)
print('Generation seconds:',round(GEN_SECONDS,1),'Finish reason:',answer.finish_reason)


## 5. Download results, then stop the runtime
Note your ending compute-unit balance and any errors. The ZIP contains the article, report and exact input. Send it back for factual/editorial review. Then use **Runtime → Disconnect and delete runtime**.

In [ ]:
import shutil
END_UNITS = None  # optionally enter the balance shown now
report['ending_units'] = END_UNITS
report['actual_units_used'] = START_UNITS-END_UNITS if END_UNITS is not None else None
(output_dir/'report.json').write_text(json.dumps(report,indent=2))
bundle = '/content/project_mu_colab_result_v' + str(job.get('dossier_version',1))
shutil.make_archive(bundle,'zip',output_dir)
files.download(bundle+'.zip')
